## Installation steps

In [6]:
# Setup: install (if needed) and download the NLTK data this notebook uses
import sys, subprocess, importlib.util
for pkg in ["nltk", "scikit-learn", "pandas"]:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import nltk
for res in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(res, quiet=True)

import os
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
print("NLTK", nltk.__version__, "ready.")

NLTK 3.9.1 ready.


## Step 1 — Load the data

Load the 10 titles and documents, confirm there are 10, and check the word count of each (both a plain `.split()` count and an NLTK word-token count).

In [7]:
# Find the repo root (the folder that contains data/documents.py) so src/ and data/ import
def find_repo_root(start=os.getcwd()):
    path = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(path, "data", "documents.py")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return None
        path = parent

REPO_ROOT = find_repo_root()
if REPO_ROOT:
    sys.path.insert(0, REPO_ROOT)
    from src.load_data import load_documents
    titles, documents = load_documents()
else:                                   # fallback: documents.py next to this notebook
    from documents import TITLES as titles, DOCUMENTS as documents

print(f"{len(documents)} documents loaded.\n")
print(f"{'Title':<22}{'split() words':>14}{'NLTK words':>12}")
for t, d in zip(titles, documents):
    nltk_words = [w for w in word_tokenize(d) if w.isalpha()]
    print(f"{t:<22}{len(d.split()):>14}{len(nltk_words):>12}")

assert len(documents) == 10
assert all(len(d.split()) > 30 for d in documents)   # each is "long text"

10 documents loaded.

Title                  split() words  NLTK words
AI in Healthcare                  85          85
Climate Change                    83          81
Basketball Strategy               85          84
Italian Pasta                     91          91
Space Exploration                 86          86
Cybersecurity                     88          87
Music Theory                      87          87
Gardening Vegetables              89          89
Economics Inflation               89          86
Public Health                     91          89


### Step 1b — NLTK preprocessing

One function turns a raw string into clean tokens. The **same function** is used for the documents and for the queries in Step 4, so both live in the same vocabulary.

1. lowercase, and split hyphenated words (`trade-offs` → `trade offs`)
2. `word_tokenize`
3. keep alphabetic tokens of 3+ letters
4. drop NLTK English stop words
5. lemmatize with WordNet (with a small exception list — WordNet would turn *Mars* into *mar*)

In [8]:
STOP_WORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
KEEP_AS_IS = {"mars", "goods"}          # words WordNet lemmatizes wrongly for this corpus

def nltk_preprocess(text):
    text = text.lower().replace("-", " ")
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and len(t) > 2]
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return [t if t in KEEP_AS_IS else lemmatizer.lemmatize(t) for t in tokens]

print("NLTK stop-word list size:", len(STOP_WORDS))
print("\nRaw (first 160 chars):\n ", documents[1][:160], "...")
print("\nAfter nltk_preprocess (first 20 tokens):\n ", nltk_preprocess(documents[1])[:20])

NLTK stop-word list size: 198

Raw (first 160 chars):
  Climate change is driven by greenhouse gas emissions from burning fossil fuels. Renewable energy sources such as solar, wind, and hydropower can reduce carbon p ...

After nltk_preprocess (first 20 tokens):
  ['climate', 'change', 'driven', 'greenhouse', 'gas', 'emission', 'burning', 'fossil', 'fuel', 'renewable', 'energy', 'source', 'solar', 'wind', 'hydropower', 'reduce', 'carbon', 'pollution', 'government', 'business']


## Step 2 — Build the TF-IDF matrix

`TfidfVectorizer` is given `nltk_preprocess` as its tokenizer, so NLTK decides what a term is and scikit-learn does the TF × IDF weighting and L2 normalisation. (`lowercase=False` and `token_pattern=None` because our function already handles both.)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(tokenizer=nltk_preprocess, lowercase=False, token_pattern=None)
X = vectorizer.fit_transform(documents)
vocab = vectorizer.get_feature_names_out()

print("Matrix shape:", X.shape)          # (10, n_unique_terms)
print("Vocabulary size:", len(vocab))

# For comparison: the original sklearn-only setup from the repo notebook
baseline = TfidfVectorizer(stop_words="english").fit(documents)
print("Vocabulary size with sklearn stop_words='english' (no NLTK):",
      len(baseline.get_feature_names_out()))

Matrix shape: (10, 527)
Vocabulary size: 527
Vocabulary size with sklearn stop_words='english' (no NLTK): 537


## Step 3 — Inspect the matrix as a DataFrame

In [10]:
df = pd.DataFrame(X.toarray(), columns=vocab, index=titles)

print("Sample of the TF-IDF matrix (first 3 docs × 10 terms):")
df.iloc[:3, :10].round(3)

Sample of the TF-IDF matrix (first 3 docs × 10 terms):


,ability,accelerate,access,accountability,accuracy,accurate,across,action,activity,adaptation
AI in Healthcare,0.000,0.0,0.0,0.132,0.132,0.000,0.0,0.000,0.0,0.000
Climate Change,0.000,0.0,0.0,0.000,0.000,0.124,0.0,0.124,0.0,0.124
Basketball Strategy,0.123,0.0,0.0,0.000,0.000,0.000,0.0,0.000,0.0,0.000


In [11]:
def top_terms(frame, n=5):
    return {t: frame.loc[t].sort_values(ascending=False).head(n) for t in frame.index}

print("Top 5 TF-IDF terms per document:")
for title, top5 in top_terms(df).items():
    print(f"\n{title}:")
    for term, score in top5.items():
        print(f"  {term:<18s} {score:.3f}")

Top 5 TF-IDF terms per document:

AI in Healthcare:
  healthcare         0.263
  use                0.174
  equitable          0.132
  replace            0.132
  require            0.132

Climate Change:
  emission           0.249
  change             0.249
  energy             0.211
  climate            0.211
  rise               0.124

Basketball Strategy:
  shot               0.245
  game               0.245
  team               0.245
  defense            0.209
  star               0.123

Italian Pasta:
  sauce              0.514
  pasta              0.411
  different          0.175
  ingredient         0.103
  tube               0.103

Space Exploration:
  space              0.350
  mars               0.233
  mission            0.233
  device             0.117
  developing         0.117

Cybersecurity:
  cybersecurity      0.242
  threat             0.242
  system             0.180
  data               0.160
  attacker           0.121

Music Theory:
  rhythm             0.240
  mus

## Step 4 — Query similarity challenge

The query goes through `vectorizer.transform`, which runs the same `nltk_preprocess`, so *"healthy sleep exercise and diet"* becomes `['healthy', 'sleep', 'exercise', 'diet']` before it is compared with every document by cosine similarity.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

def rank_documents(query, vectorizer=vectorizer, matrix=X, titles=titles, show=True):
    # Return [(title, score)] sorted by cosine similarity, highest first
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, matrix).flatten()
    ranking = sorted(zip(titles, scores), key=lambda x: x[1], reverse=True)
    if show:
        print(f"\n=== Query: {query!r} ===")
        print("  NLTK tokens:", nltk_preprocess(query))
        for title, score in ranking:
            print(f"  {title:<25s} {score:.3f}")
    return ranking

EXPECTED = {
    "renewable energy and climate policy": "Climate Change",
    "password security and phishing": "Cybersecurity",
    "healthy sleep exercise and diet": "Public Health",
}
results = {q: rank_documents(q) for q in EXPECTED}


=== Query: 'renewable energy and climate policy' ===
  NLTK tokens: ['renewable', 'energy', 'climate', 'policy']
  Climate Change            0.319
  Economics Inflation       0.122
  Gardening Vegetables      0.049
  Public Health             0.037
  AI in Healthcare          0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Cybersecurity             0.000
  Music Theory              0.000

=== Query: 'password security and phishing' ===
  NLTK tokens: ['password', 'security', 'phishing']
  Cybersecurity             0.210
  AI in Healthcare          0.000
  Climate Change            0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Music Theory              0.000
  Gardening Vegetables      0.000
  Economics Inflation       0.000
  Public Health             0.000

=== Query: 'healthy sleep exercise and diet' ===
  NLTK tokens: ['healthy', 'sleep', 'exercise', 'diet']

In [13]:
summary = pd.DataFrame(
    [(q, results[q][0][0], round(results[q][0][1], 3), EXPECTED[q], results[q][0][0] == EXPECTED[q])
     for q in EXPECTED],
    columns=["query", "ranked first", "score", "expected", "correct"],
)
summary

,query,ranked first,score,expected,correct
0,renewable energy and climate policy,Climate Change,0.319,Climate Change,True
1,password security and phishing,Cybersecurity,0.210,Cybersecurity,True
2,healthy sleep exercise and diet,Public Health,0.282,Public Health,True


### Cross-check: TF-IDF computed entirely in NLTK

NLTK has its own TF-IDF in `nltk.text.TextCollection`. Its formula is the textbook one, so the numbers differ from scikit-learn's smoothed version:

- **TF** = count of term in doc ÷ number of tokens in doc
- **IDF** = log(N ÷ number of docs containing the term) — no smoothing, so a term in every document gets 0

Below the matrix is built with `TextCollection.tf_idf`, each row is L2-normalised, and the same three queries are ranked with a dot product (= cosine similarity).

In [14]:
from nltk.text import TextCollection

doc_tokens = [nltk_preprocess(d) for d in documents]
collection = TextCollection(doc_tokens)
terms = sorted(set(t for doc in doc_tokens for t in doc))

nltk_matrix = np.array([[collection.tf_idf(term, doc) for term in terms] for doc in doc_tokens])
nltk_matrix = nltk_matrix / np.linalg.norm(nltk_matrix, axis=1, keepdims=True)
df_nltk = pd.DataFrame(nltk_matrix, columns=terms, index=titles)
print("Pure-NLTK TF-IDF matrix shape:", df_nltk.shape)

def rank_documents_nltk(query):
    q_tokens = nltk_preprocess(query)
    q_vec = np.array([q_tokens.count(t) / len(q_tokens) * collection.idf(t) for t in terms])
    if np.linalg.norm(q_vec) > 0:
        q_vec = q_vec / np.linalg.norm(q_vec)
    scores = df_nltk.values @ q_vec
    return sorted(zip(titles, scores), key=lambda x: x[1], reverse=True)

for q, expected in EXPECTED.items():
    best, score = rank_documents_nltk(q)[0]
    print(f"{q!r:42s} -> {best:<16s} ({score:.3f})  {'✓' if best == expected else '✗'}")

Pure-NLTK TF-IDF matrix shape: (10, 527)
'renewable energy and climate policy'      -> Climate Change   (0.283)  ✓
'password security and phishing'           -> Cybersecurity    (0.220)  ✓
'healthy sleep exercise and diet'          -> Public Health    (0.297)  ✓


## Step 5 — Optional challenge: bigrams

Re-run with `ngram_range=(1, 2), max_df=0.8, min_df=2`. Because the bigrams are built from NLTK's *cleaned* tokens, *"trade-offs"* becomes the bigram `trade offs`.

In [15]:
vectorizer_v2 = TfidfVectorizer(
    tokenizer=nltk_preprocess, lowercase=False, token_pattern=None,
    ngram_range=(1, 2), max_df=0.8, min_df=2,
)
X_v2 = vectorizer_v2.fit_transform(documents)
vocab_v2 = vectorizer_v2.get_feature_names_out()
df_v2 = pd.DataFrame(X_v2.toarray(), columns=vocab_v2, index=titles)

print("v1 vocabulary size (unigrams, min_df=1):", len(vocab))
print("v2 Matrix shape:", X_v2.shape)
print("v2 Vocabulary size:", len(vocab_v2))
print("Bigrams kept:", [t for t in vocab_v2 if " " in t])

v1 vocabulary size (unigrams, min_df=1): 527
v2 Matrix shape: (10, 82)
v2 Vocabulary size: 82
Bigrams kept: ['economic growth', 'policy involves', 'trade offs']


In [16]:
# Top terms side by side: v1 (all unigrams) vs v2 (shared uni + bigrams only)
v1_top, v2_top = top_terms(df), top_terms(df_v2)
pd.DataFrame({
    "v1 top-5 (unigrams)": [", ".join(v1_top[t].index) for t in titles],
    "v2 top-5 (min_df=2, bigrams)": [
        ", ".join(v2_top[t][v2_top[t] > 0].index) or "(no shared terms)" for t in titles
    ],
}, index=titles)

,v1 top-5 (unigrams),"v2 top-5 (min_df=2, bigrams)"
AI in Healthcare,"healthcare, use, equitable, replace, require","use, careful, intelligence, population, rather"
Climate Change,"emission, change, energy, climate, rise","climate, energy, economic, policy involves, scale"
Basketball Strategy,"shot, game, team, defense, star","defense, adjust, decision, based, communication"
Italian Pasta,"sauce, pasta, different, ingredient, tube","different, add, quality, rather, shape"
Space Exploration,"space, mars, mission, device, developing","cost, human, international, life, produce"
Cybersecurity,"cybersecurity, threat, system, data, attacker","system, data, intelligence, plan, regular"
Music Theory,"rhythm, music, chord, musician, practice","practice, different, track, produce, scale"
Gardening Vegetables,"soil, year, pest, garden, improves","improves, add, climate, reduces, local"
Economics Inflation,"spending, inflation, government, economic, trade","economic, government, trade, policy, decision"
Public Health,"health, public, sleep, strongly, supportive","health, clean, community, regular, shape"


In [17]:
# NLTK's own view: which bigrams repeat anywhere in the corpus?
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures

finder = BigramCollocationFinder.from_documents(doc_tokens)   # no bigrams across document borders
finder.apply_freq_filter(2)                                    # seen at least twice
for (w1, w2), pmi in finder.score_ngrams(BigramAssocMeasures().pmi):
    in_docs = [t for t, toks in zip(titles, doc_tokens)
               if any(a == w1 and b == w2 for a, b in nltk.bigrams(toks))]
    print(f"{w1 + ' ' + w2:<18s} count={finder.ngram_fd[(w1, w2)]}  PMI={pmi:5.2f}  in: {in_docs}")

economic growth    count=2  PMI= 7.82  in: ['Climate Change', 'Economics Inflation']
trade offs         count=2  PMI= 7.82  in: ['Climate Change', 'Economics Inflation']
policy involves    count=2  PMI= 7.41  in: ['Climate Change', 'Economics Inflation']
public health      count=2  PMI= 6.82  in: ['Public Health']


**Do bigrams make the terms more meaningful?** Individually, yes — `economic growth`, `public health` or `trade offs` say more than `economic`, `public` or `trade`. But with only 10 short documents on 10 different topics, `min_df=2` throws away almost everything: a term has to appear in at least two documents, so most topic-specific words and all but three bigrams disappear (the vocabulary drops from 527 to 82), and what survives are generic cross-topic words like `data`, `use` or `different`. NLTK's collocation finder confirms it: only four bigrams occur twice in the whole corpus. On this corpus the v2 settings make the summaries *less* distinctive. Bigrams pay off on larger collections where the same phrase recurs across many documents; `min_df=1` with bigrams (or the NLTK collocations above) is the better fit here.

## Step 6 — Verify work

The same three checks as `tests/test_tfidf_utils.py`, run against the NLTK pipeline, then the repo's own `pytest -q`.

In [18]:
# 1) 10 documents, each "long text"
assert len(titles) == 10 and len(documents) == 10
assert all(len(d.split()) > 30 for d in documents)
# 2) 10 rows, columns == vocabulary size
assert X.shape[0] == 10 and X.shape[1] == len(vectorizer.get_feature_names_out())
# 3) the climate query ranks Climate Change first (plus the other two queries)
for q, expected in EXPECTED.items():
    assert rank_documents(q, show=False)[0][0] == expected, q
print("All NLTK checks passed ✓")

All NLTK checks passed ✓


In [19]:
# The repo's unit tests (only runs when this notebook sits inside the cloned repo)
if REPO_ROOT and os.path.isdir(os.path.join(REPO_ROOT, "tests")):
    out = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no"], cwd=REPO_ROOT,
                         capture_output=True, text=True)
    print(out.stdout[-500:] or out.stderr[-500:])
else:
    print("tests/ folder not found — run `pytest -q` from the repo root.")

tests/ folder not found — run `pytest -q` from the repo root.


## Exit ticket

1. **What does a high TF-IDF score mean for a term in one document?** The term appears often in that document but in few others, so it is characteristic of it — e.g. `pasta` or `sauce` in *Italian Pasta*.
2. **Why does IDF reduce the weight of common words like "the" or "data"?** IDF is log(N ÷ document frequency); a word found in many documents gets a small IDF, so it contributes little. (*"the"* is already removed by NLTK's stop-word list; *"data"* appears in 4 of the 10 documents, so it is down-weighted.)
3. **Name one real task where TF-IDF is useful.** Search / document retrieval — exactly what Step 4 does — as well as keyword extraction, spam filtering and features for text classification.

**Limitation to remember:** TF-IDF matches words, not meaning. *"diet"* and *"nutrition"* are unrelated to it, and word order is ignored. Lemmatizing with NLTK helps a little (*emissions* = *emission*), but it is still a bag of words.